# Datasets [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aiboxlab/nlp/blob/main/examples/dataset.ipynb)

A biblioteca `aibox-nlp` disponibiliza datasets e corpus em Português Brasileiro. Os datasets e corpus disponibilizados se encontram no pacote `aibox.nlp.data`, é possível instanciá-los diretamente através das classes.

In [ ]:
try:
    import google.colab

    %pip install --upgrade pip uv
    !python -m uv pip install --reinstall aibox-nlp
    print("\033[31m [aibox.nlp] Reinicie o ambiente de execução antes de prosseguir.")
    exit(0)
except ImportError:
    pass

In [ ]:
from aibox.nlp.data.datasets import DatasetEssayBR, DatasetPortugueseNarrativeEssays

In [ ]:
# === Essay-BR ===
ds = DatasetEssayBR(extended=False, target_competence="C1")
ds.to_frame().head(5)

In [ ]:
# === Essay-BR Estendido ===
ds = DatasetEssayBR(extended=True, target_competence="C1")
ds.to_frame().head(5)

In [ ]:
# === Portuguese Narrative Essays ===
ds = DatasetPortugueseNarrativeEssays(target_competence="formal_register")
ds.to_frame().head(5)

In [ ]:
# Também é possível carregar os dados crus de ambos datasets
DatasetEssayBR.load_raw(extended=True).head(5)

In [ ]:
DatasetPortugueseNarrativeEssays.load_raw().head(5)

## Como utilizar um dataset próprio?

Para utilizar um dataset próprio, só precisamos carregá-lo como um `DataFrame` e utilizar a classe `nlpbox.data.datasets.DatasetDF`.

In [ ]:
import pandas as pd

from aibox.nlp.data.datasets import DatasetDF

In [ ]:
# === Exemplo de um Dataset do Kaggle ===
# A biblioteca possui o `kagglehub` instalado por padrão.
import kagglehub
from kagglehub import KaggleDatasetAdapter

df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "leandrodoze/tweets-from-mgbr",
    "Tweets_Mg.csv",
)

df.head(5)

In [ ]:
# Realizando um pequeno tratamento nos dados
df = df[["Text", "Classificacao"]].dropna()

# Normalização do texto
df["Text"] = df["Text"].str.lower()

# Remoção de caracteres especiais
df["Text"] = df["Text"].str.replace(r"[\w]*[^\w\s][\w]*", "", regex=True)
# Removendo possíveis NA
df = df.dropna()

# Removendo duplicados
df = df.drop_duplicates("Text")

# Convertendo coluna de classificação
df["Classificacao"] = (
    df["Classificacao"]
    .replace(["Positivo", "Neutro", "Negativo"], [0, 1, 2])
    .astype(int)
)

# Exemplos
df.head(5)

In [ ]:
# === Criando Dataset ===
ds = DatasetDF(df, text_column="Text", target_column="Classificacao")
ds.to_frame().target.hist()

In [ ]:
# === Obtendo splits de treino e teste ===
train, test = ds.train_test_split(0.8, True, 42)

In [ ]:
test.target.hist()

In [ ]:
train.target.hist()